In [1]:
!pip install transformers torch

In [2]:
from transformers import pipeline

# Load zero-shot classification model
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# ESG candidate labels
labels = ["Environmental", "Accessibility", "Governance", "Procurement", "Social", "Facilities"]

# 3 ESG messages to classify
messages = [
    "There is a water leak in Building C that has been running all morning.",
    "The recycling bins are contaminated again and no one seems to be checking them.",
    "The accessible entrance near the main building has been blocked for two days."
]

# Run classification
print("=" * 60)
print("HUGGING FACE ZERO-SHOT CLASSIFIER RESULTS")
print("Model: facebook/bart-large-mnli")
print("=" * 60)

for i, message in enumerate(messages):
    result = classifier(message, labels)
    top_label = result['labels'][0]
    top_score = result['scores'][0]

    print(f"\nMessage {i+1}: {message}")
    print(f"Top Category: {top_label}")
    print(f"Confidence: {top_score:.2%}")
    print("All scores:")
    for label, score in zip(result['labels'], result['scores']):
        print(f"  {label}: {score:.2%}")
    print("-" * 40)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

HUGGING FACE ZERO-SHOT CLASSIFIER RESULTS
Model: facebook/bart-large-mnli

Message 1: There is a water leak in Building C that has been running all morning.
Top Category: Facilities
Confidence: 40.71%
All scores:
  Facilities: 40.71%
  Environmental: 23.17%
  Accessibility: 17.19%
  Social: 8.85%
  Governance: 8.18%
  Procurement: 1.92%
----------------------------------------

Message 2: The recycling bins are contaminated again and no one seems to be checking them.
Top Category: Environmental
Confidence: 41.21%
All scores:
  Environmental: 41.21%
  Facilities: 18.60%
  Social: 16.81%
  Accessibility: 13.90%
  Governance: 6.92%
  Procurement: 2.56%
----------------------------------------

Message 3: The accessible entrance near the main building has been blocked for two days.
Top Category: Accessibility
Confidence: 86.12%
All scores:
  Accessibility: 86.12%
  Facilities: 6.78%
  Social: 2.80%
  Governance: 2.43%
  Environmental: 1.20%
  Procurement: 0.68%
----------------------------

In [4]:
import json

# ============================================================
# IMPROVED PROMPT TEMPLATE (Q3a)
# ============================================================
improved_prompt = """
You are an ESG operations triage analyst. Analyse the message and return ONLY valid JSON.

Fields:
- issue_category: [Environmental, Social, Governance, Facilities, Procurement, Accessibility]
- urgency: LOW, MEDIUM, HIGH, or CRITICAL
- sentiment: POSITIVE, NEUTRAL, or NEGATIVE
- followup_required: Y or N
- recommended_team: most appropriate team
- escalation_reason: brief reason or "No escalation required"
- data_sensitivity_risk: LOW, MEDIUM, or HIGH
- brief_summary: one sentence summary

Urgency guide:
- CRITICAL: immediate safety/legal risk
- HIGH: significant ongoing damage or barrier
- MEDIUM: recurring issue, moderate impact
- LOW: minor, one-off issue

If ambiguous, set urgency to MEDIUM.
"""

print("=" * 60)
print("IMPROVED PROMPT TEMPLATE")
print("=" * 60)
print(improved_prompt)

# ============================================================
# LLM (CLAUDE) OUTPUT — Q3(a)
# ============================================================
llm_results = [
    {
        "message": "There is a water leak in Building C that has been running all morning.",
        "output": {
            "issue_category": "Environmental",
            "urgency": "HIGH",
            "sentiment": "NEGATIVE",
            "followup_required": "Y",
            "recommended_team": "Facilities Management",
            "escalation_reason": "Active water leak represents ongoing environmental waste and potential property damage.",
            "data_sensitivity_risk": "LOW",
            "brief_summary": "An ongoing water leak in Building C requires urgent facilities intervention to prevent waste and damage."
        }
    },
    {
        "message": "The recycling bins are contaminated again and no one seems to be checking them.",
        "output": {
            "issue_category": "Environmental",
            "urgency": "MEDIUM",
            "sentiment": "NEGATIVE",
            "followup_required": "Y",
            "recommended_team": "ESG Team",
            "escalation_reason": "Recurring contamination suggests systemic waste management oversight failure.",
            "data_sensitivity_risk": "LOW",
            "brief_summary": "Recycling bins are repeatedly contaminated with no apparent monitoring or corrective action."
        }
    },
    {
        "message": "The accessible entrance near the main building has been blocked for two days.",
        "output": {
            "issue_category": "Accessibility",
            "urgency": "HIGH",
            "sentiment": "NEGATIVE",
            "followup_required": "Y",
            "recommended_team": "Accessibility Officer",
            "escalation_reason": "Blocked accessible entrance may breach Disability Discrimination Act 1992 obligations.",
            "data_sensitivity_risk": "LOW",
            "brief_summary": "Accessible entrance blocked for two days creating a barrier for people with disabilities."
        }
    }
]

print("\n" + "=" * 60)
print("LLM (CLAUDE) JSON OUTPUTS — Q3(a)")
print("=" * 60)
for i, item in enumerate(llm_results):
    print(f"\nMessage {i+1}: {item['message']}")
    print(json.dumps(item['output'], indent=2))
    print("-" * 40)

# ============================================================
# COMPARISON TABLE — Q3(b)
# ============================================================
hf_results = [
    {"category": "Facilities", "confidence": "40.71%"},
    {"category": "Environmental", "confidence": "41.21%"},
    {"category": "Accessibility", "confidence": "86.12%"}
]

print("\n" + "=" * 60)
print("COMPARISON: LLM vs HUGGING FACE BASELINE — Q3(b)")
print("=" * 60)
print(f"\n{'Message':<15} {'HF Category':<18} {'HF Confidence':<18} {'LLM Category':<18} {'LLM Urgency':<12} {'Match?'}")
print("-" * 95)

messages_short = ["Water leak", "Recycling bins", "Accessible entrance"]
for i in range(3):
    hf = hf_results[i]
    llm = llm_results[i]['output']
    match = "✓ YES" if hf['category'].lower() == llm['issue_category'].lower() else "✗ NO"
    print(f"{messages_short[i]:<15} {hf['category']:<18} {hf['confidence']:<18} {llm['issue_category']:<18} {llm['urgency']:<12} {match}")

print("\nKey Observations:")
print("1. Water leak: HF classified as 'Facilities' (40.71%) vs LLM 'Environmental' — LLM more precise")
print("2. Recycling: Both agree on 'Environmental' — consistent classification")
print("3. Accessibility: Both agree — HF shows high confidence (86.12%), LLM adds urgency and escalation")
print("4. LLM adds value: urgency level, escalation reason, recommended team — not possible with HF baseline")
print("5. HF limitation: low confidence on Messages 1 and 2 suggests model uncertainty")

IMPROVED PROMPT TEMPLATE

You are an ESG operations triage analyst. Analyse the message and return ONLY valid JSON.

Fields:
- issue_category: [Environmental, Social, Governance, Facilities, Procurement, Accessibility]
- urgency: LOW, MEDIUM, HIGH, or CRITICAL
- sentiment: POSITIVE, NEUTRAL, or NEGATIVE
- followup_required: Y or N
- recommended_team: most appropriate team
- escalation_reason: brief reason or "No escalation required"
- data_sensitivity_risk: LOW, MEDIUM, or HIGH
- brief_summary: one sentence summary

Urgency guide:
- CRITICAL: immediate safety/legal risk
- HIGH: significant ongoing damage or barrier
- MEDIUM: recurring issue, moderate impact
- LOW: minor, one-off issue

If ambiguous, set urgency to MEDIUM.


LLM (CLAUDE) JSON OUTPUTS — Q3(a)

Message 1: There is a water leak in Building C that has been running all morning.
{
  "issue_category": "Environmental",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "recommended_team": "Facilities 